In [ ]:
from pathlib import Path
import numpy as np
import torch

from data_utils import seed_everything, build_datasets
from evaluation import evaluate_model
from Models.AF_mamba import AFMamba
from Models.baselines import TCN_LSTMModel, TCN_OnlyModel, TCN_TransformerModel
from Models.other_models import CNNBiLSTM, CNNBiGRU, VanillaMamba
from Models.third_party.resnet1d import ResNet1D
from tsai.models.TCN import TCN
from tsai.models.InceptionTime import InceptionTime
from tsai.models.TransformerModel import TransformerModel

MODEL_NAME = "af_mamba" # Select the model for evaluation.
DATA_PATH = Path("Data/structured_dataset_1hz.pt")
FOLD_PATH = Path("Data/subject_folds.pt")
CHECKPOINT_ROOT = Path("Trained_Models")
BATCH_SIZE = 16
INPUT_SIZE = 3600
PREDICTION_HORIZON = 3600
SEED = 42

MODEL_REGISTRY = {
    "af_mamba": lambda: AFMamba(),
    "tcn_transformer": lambda: TCN_TransformerModel(),
    "tcn_bilstm": lambda: TCN_LSTMModel(),
    "tcn_baseline": lambda: TCN_OnlyModel(),
    "cnn_bilstm": lambda: CNNBiLSTM(),
    "cnn_bigru": lambda: CNNBiGRU(),
    "inceptiontime": lambda: InceptionTime(c_in=1, c_out=2),
    "resnet1d": lambda: ResNet1D(in_channels=1, base_filters=64, kernel_size=16, stride=2, groups=1, n_block=12, n_classes=2, use_bn=True, use_do=True),
    "vanilla_transformer": lambda: TransformerModel(c_in=1, c_out=2),
    "vanilla_mamba": lambda: VanillaMamba(),
    "vanilla_tcn": lambda: TCN(c_in=1, c_out=2),
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)

data = torch.load(DATA_PATH, weights_only=False)
fold_data = torch.load(FOLD_PATH, weights_only=False)
af_sets, nsr_sets = fold_data["af_sets"], fold_data["nsr_sets"]
results = []

for fold_idx in range(5):
    print(f"\n===== FOLD {fold_idx} =====")
    _, val_loader, test_loader, *_ = build_datasets(
        data, af_sets, nsr_sets, fold_idx,
        input_segment_size=INPUT_SIZE,
        prediction_horizon=PREDICTION_HORIZON,
        batch_size=BATCH_SIZE
    )

    model = MODEL_REGISTRY[MODEL_NAME]().to(device)
    checkpoint = CHECKPOINT_ROOT / MODEL_NAME / f"fold_{fold_idx}.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))

    val_metrics, *_ = evaluate_model(model, val_loader, device=device, threshold=None)
    threshold = val_metrics["threshold"]
    test_metrics, *_ = evaluate_model(model, test_loader, device=device, threshold=threshold)

    # Use evaluate_model_subject(..., subject_ids=val_sids/test_sids) for subject-level evaluation.

    print(f"Validation AUROC={val_metrics['roc_auc']:.4f}, Threshold={threshold:.4f}")
    print(
        f"Test Sens={test_metrics['recall']:.4f}, "
        f"Spec={test_metrics['specificity']:.4f}, "
        f"F1={test_metrics['f1']:.4f}, "
        f"AUROC={test_metrics['roc_auc']:.4f}, "
        f"AUPRC={test_metrics['auprc']:.4f}"
    )
    results.append(test_metrics)

metrics = {
    "Sensitivity": "recall",
    "Specificity": "specificity",
    "Precision": "precision",
    "F1": "f1",
    "AUROC": "roc_auc",
    "AUPRC": "auprc"
}

print("\n===== 5-FOLD RESULTS =====")
for name, key in metrics.items():
    values = np.array([r[key] for r in results])
    print(f"{name}: {values.mean():.4f} ± {values.std():.4f}")


===== FOLD 0 =====

=== FOLD 0 ===
Subjects: train=140, val=46, test=46
Validation AUROC=0.9412, Threshold=0.1762
Test Sens=0.9000, Spec=0.8755, F1=0.7031, AUROC=0.9611, AUPRC=0.8927

===== FOLD 1 =====

=== FOLD 1 ===
Subjects: train=140, val=46, test=46
Validation AUROC=0.9887, Threshold=0.5071
Test Sens=0.8444, Spec=0.9275, F1=0.7451, AUROC=0.9489, AUPRC=0.8930

===== FOLD 2 =====

=== FOLD 2 ===
Subjects: train=139, val=47, test=46
Validation AUROC=0.9936, Threshold=0.1819
Test Sens=0.9048, Spec=0.9955, F1=0.9421, AUROC=0.9902, AUPRC=0.9788

===== FOLD 3 =====

=== FOLD 3 ===
Subjects: train=138, val=47, test=47
